# Static FY0 report probes and CSI300 RankIC

This notebook is an explicitly conflicting alternative to the strict PIT walk-forward protocol. It uses all valid FY0 report labels in fixed feature-date splits, does not filter on `label_available_date`, opens validation and test in one Run All, and must not be interpreted as a closed confirmatory test. All business logic lives in `src/layer_probe_static.py`.

In [ ]:
import json
import os
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
assert (ROOT / 'src').is_dir(), f'Cannot locate repository root from {Path.cwd()}'
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_yaml_config
from src.layer_probe_continuous import (
    run_fixed_head_analysis_stage,
    validate_fixed_head_analysis_outputs,
)
from src.layer_probe_representations import (
    plot_representation_layer_norms,
    run_representation_stage,
    validate_representation_artifacts,
)
from src.layer_probe_static import (
    parse_static_protocol,
    plot_static_layer_correlation,
    plot_static_rank_ic,
    run_static_csi300_evaluation_stage,
    run_static_direct_return_target_stage,
    run_static_report_probe_stage,
    run_static_target_stage,
    validate_direct_return_target_outputs,
    validate_static_csi300_evaluation_outputs,
    validate_static_report_probe_outputs,
    validate_static_target_outputs,
)

CONFIG_PATH = Path(os.environ.get(
    'LAYER_PROBE_STATIC_CONFIG',
    ROOT / 'configs' / 'layer_probe_static_fy0_csi300.yaml',
)).expanduser().resolve()
BUNDLE_CONFIG_PATH = ROOT / 'configs' / 'probe_dataset_walk_forward.yaml'
config = load_yaml_config(CONFIG_PATH)
protocol = parse_static_protocol(config)
RUN_DIR = Path(config['output']['run_directory']).expanduser().resolve()
BUNDLE_DIR = Path(config['continuous_targets']['bundle_directory']).expanduser().resolve()

assert all(config['stages'].values()), 'Run All requires every configured stage enabled'
assert config['static_protocol']['pit_label_availability_enforced'] is False
assert config['static_protocol']['test_opened_in_same_run'] is True
assert config['static_protocol']['modeled_layers'] == list(range(1, 13))
display(pd.Series({
    'repo': str(ROOT),
    'config': str(CONFIG_PATH),
    'run': str(RUN_DIR),
    'target_bundle': str(BUNDLE_DIR),
}, name='paths'))
display(pd.Series(protocol.to_dict(), name='static_alternative_protocol'))

## 0. Reuse or build the full unsplit target bundle

In [ ]:
bundle_manifest_path = BUNDLE_DIR / 'probe_dataset_metadata.json'
if not bundle_manifest_path.is_file():
    subprocess.run(
        [
            sys.executable,
            str(ROOT / 'label_engineering' / 'build_probe_dataset.py'),
            '--config',
            str(BUNDLE_CONFIG_PATH),
        ],
        check=True,
        cwd=ROOT,
    )
bundle_manifest = json.loads(bundle_manifest_path.read_text(encoding='utf-8'))
assert bundle_manifest.get('splits') == [], 'Static alternative requires the unsplit full-history bundle'
display(pd.Series(bundle_manifest['counts'], name='full_history_target_bundle'))

## 1. Canonical Layer 0–12 representation store

The existing content-addressed store is hash-validated and reused when exact. Otherwise this is the only stage that reruns RoBERTa inference.

In [ ]:
started = time.perf_counter()
representation_artifacts = run_representation_stage(config)
representation_check = validate_representation_artifacts(representation_artifacts.directory)
display(pd.Series({**representation_check, 'elapsed_minutes': (time.perf_counter() - started) / 60}))
display(plot_representation_layer_norms(representation_artifacts.directory))

## 2. Frozen-head descriptive analysis

Layer 0 remains enabled here only. It is excluded from every Ridge and return-factor output.

In [ ]:
started = time.perf_counter()
fixed_head_dir = run_fixed_head_analysis_stage(config)
fixed_head_check = validate_fixed_head_analysis_outputs(fixed_head_dir)
display(pd.Series({**fixed_head_check, 'elapsed_minutes': (time.perf_counter() - started) / 60}))

## 3. Five static FY0 continuous-label tasks

In [ ]:
started = time.perf_counter()
static_target_dir = run_static_target_stage(config)
static_target_check = validate_static_target_outputs(static_target_dir)
display(pd.Series({**static_target_check, 'elapsed_minutes': (time.perf_counter() - started) / 60}))
display(pd.read_csv(static_target_dir / 'static_target_counts.csv'))

## 4. Report-level direct five-day return-rank target

In [ ]:
started = time.perf_counter()
direct_target_dir = run_static_direct_return_target_stage(config)
direct_target_check = validate_direct_return_target_outputs(direct_target_dir)
display(pd.Series({**direct_target_check, 'elapsed_minutes': (time.perf_counter() - started) / 60}))
display(pd.read_csv(direct_target_dir / 'direct_return_target_audit.csv'))

## 5. GPU report-level Ridge, Layer 1–12

The stage first validates the accelerated solver against sklearn, then scans each layer once for all six factor sources. Validation selects alpha; Test never participates in selection.

In [ ]:
started = time.perf_counter()
probe_dir = run_static_report_probe_stage(config)
probe_check = validate_static_report_probe_outputs(probe_dir)
display(pd.Series({**probe_check, 'elapsed_minutes': (time.perf_counter() - started) / 60}))
display(pd.read_csv(probe_dir / 'selected_alphas.csv'))

## 6. Historical CSI300 RankIC and layer-factor correlations

CSI300 membership is applied only here, after report weights and predictions are fixed. Validation and Test are reported separately.

In [ ]:
started = time.perf_counter()
evaluation_dir = run_static_csi300_evaluation_stage(config)
evaluation_check = validate_static_csi300_evaluation_outputs(evaluation_dir)
display(pd.Series({**evaluation_check, 'elapsed_minutes': (time.perf_counter() - started) / 60}))
rank_ic_summary = pd.read_csv(evaluation_dir / 'rank_ic_summary.csv')
display(rank_ic_summary.sort_values(['split', 'task_id', 'layer']))
display(plot_static_rank_ic(evaluation_dir, split='validation'))
display(plot_static_rank_ic(evaluation_dir, split='test'))

In [ ]:
for task_id in protocol.task_ids:
    display(plot_static_layer_correlation(evaluation_dir, task_id=task_id, split='test'))
display(plot_static_layer_correlation(evaluation_dir, task_id='direct_return_rank_5d', split='test'))

display(pd.Series({
    'run_directory': str(RUN_DIR),
    'protocol_class': 'conflicting_alternative',
    'pit_label_availability_enforced': False,
    'test_opened_in_same_run': True,
    'modeled_layers': '1-12',
    'status': 'completed',
}, name='run_summary'))